In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch
import torch.nn.functional as F
import math
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, DynamicCache
from datasets import load_dataset

In [2]:
# --- 1. 配置与模型加载 ---
MODEL_NAME = "Qwen/Qwen3-30B-A3B"
# 我们将使用 'val' 数据集中的一个子集进行演示。
# 若要评估完整数据集，请移除 .select(range(20))
DATASET_NAME = "ceval/ceval-exam"
DATASET_CONFIG = "chinese_language_and_literature"
DATASET_SPLIT = "val"
NUM_SAMPLES = 20 # 演示样本数，设为 None 可运行全集

# 生成参数
MAX_THINK_TOKENS = 4096  # 思考过程最大 token 数
STEP_TOKENS = 4          # 每步生成的 token 数

print("="*20)
print(f"模型: {MODEL_NAME}")
print(f"数据集: {DATASET_NAME} ({DATASET_CONFIG})")
print("="*20 + "\n")

模型: Qwen/Qwen3-30B-A3B
数据集: ceval/ceval-exam (chinese_language_and_literature)



In [3]:
# 加载分词器和模型
# 使用 bfloat16 以提高性能，并使用 device_map="auto" 自动分配到 GPU
print("正在加载模型和分词器...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto",
    attn_implementation="flash_attention_2",
)
# 将 pad_token_id 设置为 eos_token_id 以避免警告
model.generation_config.pad_token_id = tokenizer.eos_token_id
model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None
print("模型加载完成。\n")

正在加载模型和分词器...


Loading checkpoint shards:   0%|          | 0/16 [00:00<?, ?it/s]

模型加载完成。



In [4]:
# --- 2. 加载数据集 ---
print("正在加载数据集...")
if NUM_SAMPLES:
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT).select(range(NUM_SAMPLES))
else:
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT)
print(f"数据集加载完成，共 {len(dataset)} 个样本。\n")

正在加载数据集...
数据集加载完成，共 20 个样本。



In [5]:
def calc_perplexity(logits, token_ids):
  assert logits.shape[:-1] == token_ids.shape, \
      f"Logits shape {logits.shape} does not match token_ids shape {token_ids.shape}"
  loss = F.cross_entropy(logits.view(-1, logits.size(-1)), token_ids.view(-1), reduction='none')
  perplexity = torch.exp(loss)
  return perplexity.view(token_ids.shape)


In [ ]:
# --- 3. 生成与评估 ---
correct_predictions = 0
total_predictions = 0
options = ["A", "B", "C", "D"]

# 定义思考过程的结束标记
think_stop_string = "</think>"

# 遍历数据集
for doc in tqdm(dataset, desc="正在评估"):
    # a. 构建初始提示 (到 <think> 标签为止)
    prompt_start = (
        f"{doc['question'].strip()}\n"
        f"A. {doc['A']}\n"
        f"B. {doc['B']}\n"
        f"C. {doc['C']}\n"
        f"D. {doc['D']}\n"
        f"<think>"
    )
    input_ids = tokenizer(prompt_start, return_tensors="pt").input_ids.to(model.device)

    # 初始化KV Cache和思考过程的token列表
    past_key_values: DynamicCache | None = None
    # target_past_key_values: DynamicCache | None = None
    generated_think_ids = []
    cur_total_input_ids = input_ids

    # 初始化 KV Cache，先做一次 Prefill
    with torch.no_grad():
        outputs = model(
            cur_total_input_ids[:, :-1],
            use_cache=True,
        )
        # 扔掉最后一个 token 以规避 model.generate() bug
        past_key_values = outputs.past_key_values

    # b. 分步生成 "思考" 过程
    # 循环直到达到上限或生成停止符
    for _ in tqdm(range(math.ceil(MAX_THINK_TOKENS / STEP_TOKENS)), desc="生成思考过程", leave=False):
        num_cur_input_tokens = cur_total_input_ids.shape[-1]
        with torch.no_grad():
            # 模型从头生成4个token
            spec_outputs = model.generate(
                cur_total_input_ids,
                max_new_tokens=STEP_TOKENS,
                output_logits=True,
                return_dict_in_generate=True,
                past_key_values=past_key_values,
                use_cache=True,
            )
        
        # 提取 ppl
        # `spec_outputs.sequences` 包含了输入的token，所以我们只取最后新生成的部分
        spec_new_token_ids = spec_outputs.sequences[0, num_cur_input_tokens:]
        spec_new_token_logits = torch.concat(spec_outputs.logits, dim=0)
        spec_new_token_ppls = calc_perplexity(spec_new_token_logits, spec_new_token_ids)
        base_top_k = model.config.num_experts_per_tok
        new_token_topks = torch.where(spec_new_token_ppls > 2.0, base_top_k, base_top_k - 1)
        # 最后一个 Token 始终满激活
        # new_token_topks = torch.cat([
        #     new_token_topks,
        #     torch.tensor([base_top_k], dtype=torch.long, device=model.device)
        # ])

        # 验证投机 Tokens
        with torch.no_grad():
            # 重置 KVCache
            past_key_values.crop(num_cur_input_tokens - 1)
            # 使用最后一个 token 和新生成的前 n-1 个 token 进行验证
            verify_tokens = torch.cat([
                cur_total_input_ids[:, -1:],
                spec_new_token_ids[None, :-1]
            ], dim=1)
            verify_outputs = model(
                verify_tokens,
                use_cache=True,
                past_key_values=past_key_values,
                token_top_ks=new_token_topks,
            )

        # 提取接受的 token
        # 从第一个往后比对，直到全接受或找到第一个不接受的token
        new_token_ids = []
        for i, (spec_token_id, verify_token_logits) in enumerate(zip(spec_new_token_ids, verify_outputs.logits[0])):
            # Greedy Sampling
            verify_token_id = verify_token_logits.argmax(-1)
            new_token_ids.append(verify_token_id.item())
            if spec_token_id != verify_token_id:
                # 截断KV Cache到最大接受位置
                past_key_values.crop(num_cur_input_tokens + i)
                break
        else:
            # 如果全部接受，则还推出一个 token
            # new_token_ids.append(verify_outputs.logits[0, -1].argmax().item())
            pass

        # 将新生成的token添加到思考过程中
        generated_think_ids.extend(new_token_ids)
        last_step_input_ids = torch.tensor([new_token_ids], dtype=torch.long, device=model.device)

        # 下一步的输入必须是完整句子
        cur_total_input_ids = torch.cat([cur_total_input_ids, last_step_input_ids], dim=1)

        # 检查是否已生成结束标记
        decoded_think_part = tokenizer.decode(generated_think_ids)
        if think_stop_string in decoded_think_part:
            # 如果生成了</think>，就清理并截断多余的部分
            # 例如，如果模型生成了 "</think>好的，答案是"，我们只保留到"</think>"为止
            # 重新编码以确保 token 序列的准确性
            clean_think_part = decoded_think_part.split(think_stop_string)[0] + think_stop_string
            generated_think_ids = tokenizer.encode(clean_think_part, add_special_tokens=False)
            break
    else:
        clean_think_part = decoded_think_part + think_stop_string
    
    # c. 准备最终的提示
    prompt_end = "\n答案："
    full_sequence_text = prompt_start + clean_think_part + prompt_end
    full_sequence_ids = tokenizer(full_sequence_text, return_tensors="pt").input_ids.to(model.device)

    # --- d. LogLikelihood 评测 ---
    # 我们不再生成答案，而是直接获取模型在当前上下文后，对下一个 token 的预测 logits
    with torch.no_grad():
        # 直接进行一次前向传播，而不是 .generate()
        outputs = model(full_sequence_ids)
        
        # outputs.logits 的形状是 [batch_size, sequence_length, vocab_size]
        # 我们需要的是在整个序列末尾的那个 token 的 logits
        last_token_logits = outputs.logits[0, -1, :]

    # 使用 log_softmax 将 logits 转换为对数概率，这在数值上更稳定
    log_probs = F.log_softmax(last_token_logits, dim=-1)

    # --- e. 提取并比较各选项的对数似然值 ---
    # 从完整的对数概率分布中，只抽取我们关心的 A, B, C, D 选项的 token 的值
    option_log_likelihoods = {
        option: log_probs[tokenizer.encode(option)].item()
        for option in options
    }
    
    # 选择对数似然值最高的选项作为模型的预测答案
    predicted_answer = max(option_log_likelihoods, key=option_log_likelihoods.get)
    ground_truth_answer = doc["answer"]

    # 打印单项的详细结果（可选）
    print("\n---")
    print(full_sequence_text)
    # 打印每个选项的对数似然值，便于分析
    print(f"LogLikelihoods: {option_log_likelihoods}")
    print(f"模型预测: {predicted_answer}, 正确答案: {ground_truth_answer}")
    print("---\n")

    if predicted_answer == ground_truth_answer:
        correct_predictions += 1
    total_predictions += 1

# --- 4. 输出最终结果 ---
accuracy = (correct_predictions / total_predictions) * 100 if total_predictions > 0 else 0
print("\n" + "="*20)
print("LogLikelihood 评估完成！")
print(f"总样本数: {total_predictions}")
print(f"正确数: {correct_predictions}")
print(f"准确率: {accuracy:.2f}%")
print("="*20)

正在评估:   5%|▌         | 1/20 [04:18<1:21:49, 258.40s/it]


---
英语名词“lab”(实验室)原来的形式是“laboratory”，这在词的形成方式上属于____。
A. 直接成词
B. 变形成词
C. 变性成词
D. 逆序成词
<think>
嗯，我现在要解决这个问题，题目是关于词的形成方式，具体是“lab”这个名词原来的形式是“laboratory”，问属于哪一种成词方式，选项有四个：直接成词、变形成词、变性成词、逆序成词。首先，我需要回忆一下这些术语的定义，可能需要先理解每个选项的意思。

首先，直接成词应该是指直接由一个词或词根构成新词，比如“book”变成“bookstore”，不过可能不是这种情况。或者可能是指直接使用原词，比如“lab”直接作为“laboratory”的缩写？不过题目里说“lab”原来的形式是“laboratory”，可能是指“lab”是从“laboratory”演变而来的，所以可能不是直接成词。

变形成词可能指的是通过改变词形，比如加前缀、后缀，或者改变词根，比如“laboratory”变成“labor”或者“lab”，可能属于缩略或者变形。不过变形成词可能指的是通过词形变化，比如动词变名词，或者名词变形容词之类的，但这里可能不是这种情况。

变性成词可能指的是词性发生变化，比如动词变成名词，或者名词变成动词，比如“email”作为名词和动词，但这里“laboratory”是名词，而“lab”也是名词，所以可能不是变性。

逆序成词可能是指把词的字母顺序颠倒，比如“stop”变成“pots”，但“lab”和“laboratory”显然不是逆序的，所以这个选项应该不对。

不过可能我理解错了这些术语的定义。可能需要更仔细地分析。

首先，“laboratory”缩写为“lab”，这属于缩略词，而缩略词的形成方式可能有不同的类型，比如首字母缩略词（如NASA），或者截短词（如“television”→“TV”）。这里“laboratory”变成“lab”是截短词，也就是从原词中截取一部分，通常是前面的音节或字母。这种情况下，可能属于“截短”（shortening）的一种，但题目中的选项没有这个，所以需要看选项中哪个更接近。

题目中的选项是直接成词、变形成词、变性成词、逆序成词。可能这里的“变形成词”指的是通过变形，比如缩写或截短，而“变性成词”可能指词性变化，比如名词变动词等。而“逆序成词

正在评估:  10%|█         | 2/20 [07:34<1:06:34, 221.94s/it]


---
音位是____。
A. 最小的语音语义结合体
B. 一个语音系统中能区别意义的最小语音单位
C. 语言中最小的能区别意义的语音特征单位
D. 语言中有区别性特征的最小语音单位
<think>
嗯，我现在要解决这个关于音位的问题。题目是问音位是什么，选项有四个：A、B、C、D。我需要仔细分析每个选项，然后确定正确的答案。

首先，我得回忆一下音位的定义。音位是语言学中的一个概念，属于语音学或音系学的范畴。根据以前学过的知识，音位应该是指能够区别意义的最小语音单位。不过具体怎么表述可能需要更准确。

先看选项A：“最小的语音语义结合体”。这听起来像是音节或者词，因为音节是语音和语义的结合体，但音位可能更小。比如，音位本身可能不直接承载语义，而是通过组合成音节或词来产生意义。所以A可能不对。

选项B：“一个语音系统中能区别意义的最小语音单位”。这个比较接近我之前学的定义。音位确实是在某个语言的语音系统中，能够区别意义的最小单位。比如，在英语中，/p/和/b/是不同的音位，因为它们可以区分单词，比如“pat”和“bat”。所以B看起来是对的。

选项C：“语言中最小的能区别意义的语音特征单位”。这里提到的是“语音特征单位”，而音位本身是音段（segment），而语音特征是更小的单位，比如送气、鼻化等。比如，音位是音段，而音段由语音特征组成。所以可能C的说法有问题，因为音位不是特征单位，而是由特征组成的。所以C可能不正确。

选项D：“语言中有区别性特征的最小语音单位”。这里的“区别性特征”可能指的是能区别意义的特征，但同样，音位本身是单位，而区别性特征是构成音位的属性。所以D可能也不正确，因为音位是单位，而区别性特征是其属性。

不过，可能我需要再仔细核对一下定义。比如，音位的定义是否是“能区别意义的最小语音单位”，而选项B就是这个意思。而选项C中的“语音特征单位”可能指的是音位的构成要素，比如元音和辅音的分类，或者更细的特征，比如清音、浊音等。所以，音位本身是单位，而特征是构成音位的属性。因此，选项C可能不正确，而选项B正确。

不过，有时候可能会有不同的术语使用。比如，是否有的教材中将音位定义为“最小的能区别意义的语音特征单位”？不过一般来说，音位是音段，而特征是更小的单位。例如，音位/ p /由特征如[+bilabial], [-voice], [+sto

正在评估:  15%|█▌        | 3/20 [09:28<48:55, 172.71s/it]  

...


---
《本草纲目》的作者是____李时珍。
A. 南宋人
B. 明朝人
C. 元朝人
D. 清朝人
<think>
嗯，用户问的是《本草纲目》的作者是谁，选项是李时珍，然后四个朝代选项。首先，我需要确认《本草纲目》的作者确实是李时珍，这点我记得没错。不过用户可能想确认的是李时珍所属的朝代，所以需要确定李时珍是哪个朝代的人。

李时珍是明朝的，对吧？我记得他生活在16世纪，也就是明朝中后期。可能用户需要的是这个信息，所以正确选项应该是B选项，明朝人。不过，我得再仔细想想有没有可能记错。比如，有没有可能李时珍属于其他朝代？比如，有时候可能会有混淆，比如宋朝或者清朝的医家。不过李时珍的生平应该是在明朝，他的著作《本草纲目》成书于明朝，所以应该没错。

另外，用户可能对历史人物和朝代不太熟悉，所以需要明确回答。可能用户是在做作业或者准备考试，需要准确的答案。所以需要确认正确性。再想想，李时珍的生卒年份，我记得他生于1518年，死于1593年，这属于明朝的嘉靖到万历年间，所以确实是明朝人。因此选项B正确。

不过，可能用户会误以为是其他朝代，比如有的可能以为是宋朝，因为宋朝有很多科技和医学的发展，但《本草纲目》是明朝的。或者可能混淆了《本草纲目》和《伤寒杂病论》之类的，但后者是张仲景的，东汉时期的。所以需要明确区分。

总之，答案应该是B选项，明朝人。用户可能需要这个信息来完成题目，所以直接给出正确答案即可，同时确保自己没有记错。
</think>
答案：
LogLikelihoods: {'A': -18.5, 'B': -1.3125, 'C': -21.75, 'D': -20.625}
模型预测: B, 正确答案: B
---



正在评估:  95%|█████████▌| 19/20 [1:33:57<02:57, 177.91s/it]


---
“我、你、他”是____。
A. 指示代词
B. 疑问代词
C. 人称代词
D. 名词
<think>
嗯，我现在要解决这个问题，题目是“我自己、你、他”属于哪一类词，选项是A到D。首先，我需要回忆一下汉语语法中代词的分类。代词通常分为人称代词、指示代词、疑问代词等。

题目中的例子是“我自己、你、他”。首先，“自己”通常作为反身代词，用来指代前面提到的主语，比如“我自己去”中的“自己”指的是“我”。不过有时候“自己”也可以作为指示代词，但可能更常见的是作为反身代词。不过这里可能需要更仔细分析。

然后是“你”和“他”，这两个明显是人称代词，分别指第二人称和第三人称。所以“你”和“他”属于人称代词，而“自己”可能属于反身代词，但反身代词通常也被归类为人称代词的一种。不过有时候可能被单独分类，但选项中没有反身代词这个选项，所以可能需要看选项中的分类。

题目给出的选项是：A. 指示代词，B. 疑问代词，C. 人称代词，D. 名词。所以问题在于“自己”是否属于人称代词。根据汉语语法，人称代词包括“我、你、他、我们、你们、他们”等，而“自己”通常被视为反身代词，但有时也被视为人称代词的一种。例如，在现代汉语中，反身代词可能被归类为人称代词的子类。因此，“自己”可能属于人称代词，而“你”和“他”显然属于人称代词。因此，这三个词都属于人称代词，所以正确选项是C。

不过，也有可能题目中的“自己”被单独分类，但选项中没有，所以可能需要考虑其他可能性。比如，是否“自己”是指示代词？不过指示代词通常是“这、那、这些、那些”等，而“自己”不是指示代词。疑问代词如“谁、什么、哪里”等，显然也不符合。所以“自己”应该属于人称代词，所以正确选项是C。

不过，可能有人会认为“自己”是反身代词，而题目中的选项没有这个，所以可能需要选C。或者可能题目中的“自己”被当作指示代词？不过不太对。所以综合来看，正确答案应该是C选项，人称代词。
</think>
答案：
LogLikelihoods: {'A': -14.0625, 'B': -15.5625, 'C': -0.091796875, 'D': -18.125}
模型预测: C, 正确答案: C
---



正在评估: 100%|██████████| 20/20 [1:37:26<00:00, 292.34s/it]


---
关于词的本义和基本义的关系，下列表述正确的是____。
A. 完全不一致
B. 完全一致
C. 本义比基本义更常用
D. 基本义比本义更常用
<think>
嗯，我现在要解决这个问题，题目是关于词语的本义和基本义的关系，选项是四个：A完全不一致，B完全一致，C本义比基本义更常用，D基本义比本义更常用。首先，我需要明确什么是本义，什么是基本义。

首先，本义应该是指一个词最初的意义，也就是这个词在古代文献中最早出现时的含义。比如“兵”字，本义可能是指武器或者士兵，比如在《说文解字》里可能有解释。而基本义可能是指这个词现在最常用的意义，也就是现代汉语中常用的含义。比如“兵”现在可能更多指士兵或者军队，或者战争相关的。

不过，可能有时候本义和基本义会有重叠，比如有些词的本义和现在常用的基本义是一致的。比如“水”这个字，本义可能就是指水，而基本义也是指水，所以这时候两者是一致的。但可能有些词的本义和基本义不同，比如“发”字，本义可能是指头发，而基本义可能是指发送或者发生，比如“发信”或者“发芽”。

不过题目问的是本义和基本义的关系，正确的选项应该是什么？题目给出的选项中，B选项是完全一致，D选项是基本义比本义更常用。那可能需要分析本义和基本义的定义。

根据现代汉语词汇学的知识，本义是指词的最初意义，而基本义是词的最常用的意义。有时候本义可能和基本义相同，但有时候可能不同。比如“道”字，本义可能是指道路，而基本义可能是指道理、道德等，这时候两者就不一致。所以本义和基本义可能不一致，所以A选项“完全不一致”可能不对，因为有时候可能一致。而B选项“完全一致”也不对，因为有时候不一致。那剩下的就是C和D选项，即哪个更常用。

不过问题不是问是否一致，而是它们的关系。题目问的是“正确的是哪一个”，所以可能需要更准确的定义。比如，本义和基本义是否一致？或者哪个更常用？

根据一些资料，基本义通常是指现代最常用的意义，而本义可能是指词的最初意义。有时候基本义可能来源于本义，但有时候可能演变出新的意义。例如，“兵”的本义是武器，而基本义可能指士兵或军队，所以两者可能不同。这时候，基本义可能和本义不同，所以A选项“完全不一致”可能不正确，因为可能有些词的本义和基本义是一致的，比如“日”本义是太阳，基本义也是太阳，所以这时候两者一致。所以不能说完全不一致，所以A不对。B选项“完

In [ ]:
predicted_answer


In [80]:
ground_truth_answer

'C'